# Actividad: Modelos Predictivos con Credit Risk Data

**Curso:** Analitica Predictiva y Machine Learning  
**Profesor:** Heber E. Bermudez  

---

En esta actividad vas a recorrer el flujo completo de un proyecto de prediccion:  
1. Cargar datos desde una URL publica  
2. Explorar y visualizar  
3. Construir una regresion simple  
4. Construir una regresion multiple  
5. Construir una regresion logistica  
6. **Tu turno:** elegir las variables para predecir un nuevo cliente  

> Las celdas marcadas con **TU TURNO** requieren que completes el codigo.

---
## Paso 1: Importar Librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (10, 6)

print('Librerias cargadas correctamente!')

---
## Paso 2: Cargar los Datos desde URL

In [ ]:
url = 'https://raw.githubusercontent.com/hebermudezg/hebermudezg/master/courses/statistics/credit_risk_data.csv'
df = pd.read_csv(url)

print(f'Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas')
df.head()

---
## Paso 3: Conocer los Datos

In [ ]:
print(df.describe())
print('\n--- Distribucion de estado_pago ---')
print(df['estado_pago'].value_counts(normalize=True).round(3))

---
## Paso 4: Limpieza

In [ ]:
df = df.dropna()
print(f'Filas despues de limpieza: {len(df):,}')

---
## Paso 5: Visualizacion

In [ ]:
# Correlograma
cols_num = ['edad', 'antiguedad_laboral', 'ingresos_anuales', 'puntaje_credito', 'monto_prestamo']
plt.figure(figsize=(7, 5))
sns.heatmap(df[cols_num].corr(), annot=True, fmt='.3f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Correlograma: Variables Numericas')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots por estado de pago
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, col in enumerate(['puntaje_credito', 'monto_prestamo', 'ingresos_anuales']):
    sns.boxplot(data=df, x='estado_pago', y=col, ax=axes[i],
                palette={'Pagado': '#00A99D', 'No Pagado': '#e74c3c'})
    axes[i].set_title(col.replace('_', ' ').title())
plt.tight_layout()
plt.show()

---
## Paso 6: Train / Test Split

In [ ]:
X = df[['ingresos_anuales']]
y = df['monto_prestamo']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')

---
## Paso 7: Regresion Simple

In [ ]:
modelo_simple = LinearRegression()
modelo_simple.fit(X_train, y_train)

r2_train = modelo_simple.score(X_train, y_train)
r2_test  = modelo_simple.score(X_test, y_test)

print('========== REGRESION SIMPLE ==========')
print(f'Intercepto (beta_0): ${modelo_simple.intercept_:>13,.2f} COP')
print(f'Pendiente  (beta_1): {modelo_simple.coef_[0]:>14.4f}')
print(f'R2 entrenamiento:    {r2_train:>13.3f}')
print(f'R2 prueba:           {r2_test:>13.3f}')

---
## Paso 8: Regresion Multiple

In [ ]:
X = df[['edad', 'antiguedad_laboral', 'ingresos_anuales', 'puntaje_credito']]
y = df['monto_prestamo']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_multiple = LinearRegression()
modelo_multiple.fit(X_train, y_train)

print('--- Regresion Multiple ---')
print(f'Intercepto: ${modelo_multiple.intercept_:,.2f}')
for nombre, coef in zip(X_train.columns, modelo_multiple.coef_):
    print(f'  {nombre}: {coef:.4f}')
print(f'\nR2 entrenamiento: {modelo_multiple.score(X_train, y_train):.3f}')
print(f'R2 prueba:        {modelo_multiple.score(X_test, y_test):.3f}')

---
## Paso 9: Regresion Logistica

In [ ]:
df['no_pago'] = (df['estado_pago'] == 'No Pagado').astype(int)

X = df[['puntaje_credito', 'monto_prestamo', 'ingresos_anuales', 'antiguedad_laboral']]
y = df['no_pago']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

modelo_logistico = LogisticRegression(max_iter=1000, random_state=42)
modelo_logistico.fit(X_train_s, y_train)

print(f'Accuracy entrenamiento: {modelo_logistico.score(X_train_s, y_train):.3f}')
print(f'Accuracy prueba:        {modelo_logistico.score(X_test_s, y_test):.3f}')
print(classification_report(y_test, modelo_logistico.predict(X_test_s),
                            target_names=['Pagado', 'No Pagado']))

### Matriz de Confusion

In [ ]:
y_pred = modelo_logistico.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=['Pagado', 'No Pagado']).plot(ax=ax, cmap='Blues')
ax.set_title('Matriz de Confusion - Regresion Logistica')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Accuracy:    {(tp+tn)/(tp+tn+fp+fn):.3f}')
print(f'Sensitivity: {tp/(tp+fn):.3f}')
print(f'Specificity: {tn/(tn+fp):.3f}')
print(f'Precision:   {tp/(tp+fp):.3f}')

---
---

# TU TURNO: Predice un Nuevo Cliente

Imagina que llega un nuevo cliente al banco. Tu trabajo es usar los modelos que ya entrenamos para predecir:

1. **Cuanto prestamo solicitaria** (regresion multiple)
2. **Si pagara o no** (regresion logistica)

Completa las variables del cliente con los valores que tu quieras (valores realistas).

### Actividad 1: Predecir el monto del prestamo

Completa los datos del cliente y ejecuta la celda.

In [ ]:
# ============================================
# TU TURNO: Completa los datos del cliente
# ============================================

mi_cliente = {
    'edad':                ___,   # Ejemplo: 35 (rango: 22-65)
    'antiguedad_laboral':  ___,   # Ejemplo: 10 (rango: 0-40)
    'ingresos_anuales':    ___,   # Ejemplo: 40000000 (en COP)
    'puntaje_credito':     ___,   # Ejemplo: 620 (rango: 300-850)
}

# --- No modifiques debajo de esta linea ---
cliente_df = pd.DataFrame([mi_cliente])
prediccion_monto = modelo_multiple.predict(cliente_df)[0]

print('========== PREDICCION: MONTO DEL PRESTAMO ==========')
print(f'Edad:                {mi_cliente["edad"]} anos')
print(f'Antiguedad laboral:  {mi_cliente["antiguedad_laboral"]} anos')
print(f'Ingresos anuales:    ${mi_cliente["ingresos_anuales"]:,.0f} COP')
print(f'Puntaje de credito:  {mi_cliente["puntaje_credito"]}')
print(f'\n>>> Monto estimado del prestamo: ${prediccion_monto:,.0f} COP')

### Actividad 2: Predecir si el cliente pagara o no

Usa los mismos datos del cliente anterior (o cambia los valores) para predecir el riesgo.

In [ ]:
# ============================================
# TU TURNO: Completa los datos del cliente
# ============================================

mi_cliente_riesgo = {
    'puntaje_credito':     ___,   # Ejemplo: 620 (rango: 300-850)
    'monto_prestamo':      ___,   # Ejemplo: 20000000 (en COP)
    'ingresos_anuales':    ___,   # Ejemplo: 40000000 (en COP)
    'antiguedad_laboral':  ___,   # Ejemplo: 10 (rango: 0-40)
}

# --- No modifiques debajo de esta linea ---
cliente_riesgo_df = pd.DataFrame([mi_cliente_riesgo])
cliente_riesgo_s = scaler.transform(cliente_riesgo_df)

prediccion = modelo_logistico.predict(cliente_riesgo_s)[0]
probabilidad = modelo_logistico.predict_proba(cliente_riesgo_s)[0]

print('========== PREDICCION: RIESGO DE IMPAGO ==========')
print(f'Puntaje de credito:  {mi_cliente_riesgo["puntaje_credito"]}')
print(f'Monto del prestamo:  ${mi_cliente_riesgo["monto_prestamo"]:,.0f} COP')
print(f'Ingresos anuales:    ${mi_cliente_riesgo["ingresos_anuales"]:,.0f} COP')
print(f'Antiguedad laboral:  {mi_cliente_riesgo["antiguedad_laboral"]} anos')
print(f'\n>>> Prediccion: {"NO PAGARA" if prediccion == 1 else "SI PAGARA"}')
print(f'>>> Probabilidad de pago:    {probabilidad[0]*100:.1f}%')
print(f'>>> Probabilidad de impago:  {probabilidad[1]*100:.1f}%')

### Actividad 3: Experimenta con diferentes perfiles

Prueba estos 3 perfiles y anota los resultados en la tabla:

| Perfil | Edad | Antiguedad | Ingresos | Puntaje | Monto Estimado | Pagara? | P(impago) |
|--------|------|------------|----------|---------|----------------|---------|----------|
| Joven riesgoso | 24 | 1 | 18,000,000 | 380 | ___ | ___ | ___ |
| Profesional estable | 42 | 15 | 50,000,000 | 720 | ___ | ___ | ___ |
| Senior endeudado | 55 | 25 | 60,000,000 | 450 | ___ | ___ | ___ |

Completa la tabla ejecutando las celdas anteriores con cada perfil.

In [ ]:
# TU TURNO: Copia y pega el codigo de arriba para cada perfil,
# cambia los valores y anota los resultados en la tabla.


---

### Preguntas de Reflexion

Responde brevemente:

1. **Cual variable tiene mas influencia en el monto del prestamo?** (Pista: mira los coeficientes del modelo multiple)

2. **Por que el modelo logistico necesita StandardScaler pero el lineal no?**

3. **Si un cliente tiene puntaje 850 pero pide un prestamo de $80,000,000, el modelo lo clasifica como riesgoso? Por que?**

**Tus respuestas:**

1. ...

2. ...

3. ...